# Gold Direction ML — 3-Class Profit-Maximizing Pipeline

A volatility-aware, **both-directions** machine-learning pipeline that trades gold
from a **$100,000** toy account, where the model decides *direction* and *size*.

1. **Expanded features**, then an **empirical search for the best subset**.
2. **Three-class triple-barrier labels** (López de Prado) — every bar is labeled
   **down (−1) / flat (0) / up (+1)** by whichever volatility-scaled barrier is
   touched first. The model goes **long on +1, short on −1, flat on 0**.
3. **Purged walk-forward cross-validation** with Optuna, optimized on a
   **profit × Sharpe composite** (final equity ratio × annualized Sharpe).
4. **Confidence-proportional position sizing** — bigger directional edge ⇒ larger
   fraction of current equity (compounded, **no leverage**).
5. **Model comparison** (XGBoost vs LightGBM vs logistic vs **LSTM**) by profit
   **+ SHAP** explainability.
6. **$100k backtest** reporting final account value, Sharpe / Sortino / max-drawdown,
   trade stats, and a per-regime breakdown.

> Requires network access (yfinance) to execute.

**Organization:** each cell does one focused job — a single function definition,
a single transformation, or a single plot.

## 0. Dependencies

In [ ]:
# Install extra dependencies (safe to re-run; quiet)
# tensorflow-cpu powers the LSTM baseline; statsmodels powers the ARIMA feature.
%pip install -q optuna shap lightgbm statsmodels tensorflow-cpu

## 1. Setup & Data

Forward-fill only (no back-fill lookahead), with MultiIndex handling. We include
the S&P 500 (`^GSPC`) to support cross-asset / risk-on-risk-off features.

In [ ]:
import warnings
import numpy as np
import pandas as pd
import yfinance as yf
import matplotlib.pyplot as plt
import seaborn as sns

warnings.filterwarnings("ignore", category=FutureWarning)
warnings.filterwarnings("ignore", category=UserWarning)
sns.set_theme(style="whitegrid")
plt.rcParams["figure.figsize"] = (12, 6)

In [ ]:
CONFIG = {
    # technical windows
    "rsi_window": 14, "rsi_window_short": 5,
    "macd_fast": 12, "macd_slow": 26, "macd_signal": 9,
    "sma_short": 50, "sma_long": 200,
    "vol_window": 21, "vol_window_short": 10,
    "ratio_z_window": 252, "vix_z_window": 252, "corr_window": 60,
    # LSTM comparison baseline
    "lstm_seq_len": 20, "lstm_epochs": 15,
    # triple-barrier labeling (two-sided, 3-class)
    "pt_mult": 2.0, "sl_mult": 2.0, "max_holding_days": 10,
    # walk-forward CV + search  (keep small for fast runs; raise for production)
    "n_splits": 5, "embargo": 5, "n_trials": 50, "holdout_frac": 0.20,
    # money management — the model sizes trades; objective is profit × Sharpe
    "start_capital": 100_000.0, "max_position_frac": 1.0,
    "edge_threshold": 0.10, "size_scale": 3.0,
    # backtest
    "cost_per_trade": 0.0005, "trading_days": 252,
    "random_state": 42,
}

In [ ]:
def fetch_macro_data(tickers_dict, start_date="2000-01-01", end_date="2026-01-01"):
    tickers_list = list(tickers_dict.keys())
    print(f"Fetching macro data for {tickers_list}...")
    raw_df = yf.download(tickers_list, start=start_date, end=end_date,
                         auto_adjust=True, progress=False)
    if isinstance(raw_df.columns, pd.MultiIndex):
        close_df = raw_df["Close"].copy()
    else:
        close_df = raw_df[["Close"]].copy()
    close_df = close_df.rename(columns=tickers_dict)
    # Forward-fill only: back-filling leading NaNs would leak future values into
    # the warm-up period. Warm-up rows are dropped later via dropna().
    return close_df.ffill()

In [ ]:
tickers_map = {
    "GC=F":     "Gold_Close",
    "DX-Y.NYB": "DXY_Close",
    "^VIX":     "VIX_Close",
    "^TNX":     "TNX_Close",
    "SI=F":     "Silver_Close",
    "^GSPC":    "SPX_Close",
    "TIP":      "TIP_Close",      # iShares TIPS ETF — real yield proxy (from 2004)
    "GDX":      "GDX_Close",      # VanEck Gold Miners ETF — lead indicator (from 2006)
    "CL=F":     "Oil_Close",      # crude oil futures — commodity complex
    "EURUSD=X": "EURUSD_Close",   # EUR/USD — dollar weakness signal
}

raw_data = fetch_macro_data(tickers_map)
print("Data fetched successfully.", raw_data.shape)

## 2. Feature Engineering

All features use only past data (`rolling` / `ewm` / `shift`). In addition to
the original technical indicators, this section now computes four macro factor
groups with documented predictive relationships to gold:

- **Real yields** (TIP ETF z-score + TNX-based real-rate stress) — gold's
  strongest documented macro driver; gold rallies when real yields fall.
- **Gold miners** (GDX/gold ratio z-score + miner momentum spread) — miner
  equity tends to lead physical gold by days to weeks.
- **Dollar & EUR/USD** (DXY already included; EUR/USD 21-day momentum adds the
  cross-rate angle, since dollar weakness is the primary FX driver of gold).
- **Crude oil** (21-day momentum + gold/oil ratio z-score) — oil captures
  inflation and commodity-complex demand that co-moves with gold.

Feature names are centralized in `FEATURES` so the model, SHAP, and the
backtest read from a single source of truth.

In [ ]:
def engineer_features(df, config=CONFIG):
    print("Engineering features...")
    p = df.copy().sort_index()
    c = p["Gold_Close"]
    log_ret = np.log(c / c.shift(1))

    # --- RSI (two windows) ---
    def rsi(series, window):
        delta = series.diff()
        gain = delta.clip(lower=0)
        loss = -delta.clip(upper=0)
        com = window - 1
        eg = gain.ewm(com=com, adjust=False).mean()
        el = loss.ewm(com=com, adjust=False).mean()
        return 100 - (100 / (1 + eg / (el + 1e-10)))
    p["Gold_RSI"] = rsi(c, config["rsi_window"])
    p["Gold_RSI_5"] = rsi(c, config["rsi_window_short"])

    # --- Normalized MACD histogram (scaled by price) ---
    exp1 = c.ewm(span=config["macd_fast"], adjust=False).mean()
    exp2 = c.ewm(span=config["macd_slow"], adjust=False).mean()
    macd_line = (exp1 - exp2) / c
    signal_line = macd_line.ewm(span=config["macd_signal"], adjust=False).mean()
    p["Gold_MACD_Hist"] = macd_line - signal_line

    # --- Trend ---
    sma_s = c.rolling(config["sma_short"]).mean()
    sma_l = c.rolling(config["sma_long"]).mean()
    p["Dist_SMA_50"] = (c - sma_s) / sma_s
    p["Dist_SMA_200"] = (c - sma_l) / sma_l
    p["SMA_Slope_200"] = sma_l.pct_change(periods=21)
    p["Trend_Sign"] = (sma_s > sma_l).astype(int)          # golden/death cross

    # --- Momentum (lagged log returns) ---
    for k in (1, 5, 10, 21):
        p[f"Mom_{k}"] = np.log(c / c.shift(k))

    # --- Volatility & regime ---
    p["Gold_Vol"] = log_ret.rolling(config["vol_window"]).std()
    vol_s = log_ret.rolling(config["vol_window_short"]).std()
    p["Vol_Ratio"] = vol_s / (p["Gold_Vol"] + 1e-10)
    vol_med = p["Gold_Vol"].rolling(config["ratio_z_window"]).median()
    p["High_Vol_Regime"] = (p["Gold_Vol"] > vol_med).astype(int)

    # --- Macro ---
    if "DXY_Close" in p:
        p["DXY_Pct_Change"] = p["DXY_Close"].pct_change()
    if "TNX_Close" in p:
        p["TNX_Diff"] = p["TNX_Close"].diff() / 100.0
    if "VIX_Close" in p:
        p["VIX_3d_Diff"] = p["VIX_Close"].diff(periods=3)
        vmean = p["VIX_Close"].rolling(config["vix_z_window"]).mean()
        vstd = p["VIX_Close"].rolling(config["vix_z_window"]).std()
        p["VIX_Z"] = (p["VIX_Close"] - vmean) / (vstd + 1e-10)
    if "Silver_Close" in p:
        # rolling z-score of the gold/silver ratio (stationary; trees can use it)
        ratio = c / p["Silver_Close"]
        roll = ratio.rolling(config["ratio_z_window"])
        p["Gold_Silver_Ratio_Z"] = (ratio - roll.mean()) / (roll.std() + 1e-10)
        sil_mom = np.log(p["Silver_Close"] / p["Silver_Close"].shift(21))
        p["Gold_Silver_Mom_Spread"] = p["Mom_21"] - sil_mom

    # --- Cross-asset ---
    w = config["corr_window"]
    if "SPX_Close" in p:
        spx_mom = np.log(p["SPX_Close"] / p["SPX_Close"].shift(21))
        p["Gold_SPX_Mom_Spread"] = p["Mom_21"] - spx_mom
    if "DXY_Close" in p:
        p["Corr_Gold_DXY"] = log_ret.rolling(w).corr(
            np.log(p["DXY_Close"] / p["DXY_Close"].shift(1)))
    if "TNX_Close" in p:
        p["Corr_Gold_TNX"] = log_ret.rolling(w).corr(p["TNX_Close"].diff())

    # --- Calendar (cyclical encoding: keeps Dec~Jan and Fri~Mon adjacent,
    #     which raw integer month/day-of-week would treat as maximally distant) ---
    dow = p.index.dayofweek                                 # 0=Mon .. 4=Fri
    month = p.index.month                                   # 1 .. 12
    p["DOW_sin"] = np.sin(2 * np.pi * dow / 5)              # 5-day trading week
    p["DOW_cos"] = np.cos(2 * np.pi * dow / 5)
    p["Month_sin"] = np.sin(2 * np.pi * month / 12)
    p["Month_cos"] = np.cos(2 * np.pi * month / 12)
    p["TurnOfMonth"] = ((p.index.day <= 3) | (p.index.day >= 26)).astype(int)

    # --- Real yields (TIP ETF + TNX) ---
    # Gold's strongest documented macro driver: falls when real yields rise.
    # TIP price return ≈ negative of real yield change (duration ~8 yr).
    if "TIP_Close" in p:
        tip_lr = np.log(p["TIP_Close"] / p["TIP_Close"].shift(1))
        tip_roll = tip_lr.rolling(config["vix_z_window"])
        p["TIP_Z"] = (tip_lr - tip_roll.mean()) / (tip_roll.std() + 1e-10)
        if "TNX_Close" in p:
            # positive = nominal yield surging while TIPS flat = real rate spike (bearish gold)
            tnx_std = p["TNX_Close"].rolling(config["ratio_z_window"]).std() + 1e-10
            tnx_z = p["TNX_Close"].diff() / tnx_std
            p["RealRate_Stress"] = tnx_z - p["TIP_Z"]

    # --- Gold miners (GDX, available from 2006) ---
    # Miner equity carries operational leverage to the gold price and tends to
    # lead physical gold by a few trading days; the ratio z-score captures
    # temporary dislocations (high miner premium → metal may catch up).
    if "GDX_Close" in p:
        gdx_lr = np.log(p["GDX_Close"] / p["GDX_Close"].shift(1))
        ratio_gdx = p["GDX_Close"] / c
        r_gdx = ratio_gdx.rolling(config["ratio_z_window"])
        p["GDX_Gold_Ratio_Z"] = (ratio_gdx - r_gdx.mean()) / (r_gdx.std() + 1e-10)
        p["GDX_Gold_Mom_Spread"] = gdx_lr.rolling(21).sum() - log_ret.rolling(21).sum()

    # --- Crude oil ---
    # Oil and gold share an inflation/commodity-complex driver; the gold/oil
    # ratio z-score captures safe-haven premium vs. cyclical demand shifts.
    if "Oil_Close" in p:
        p["Oil_Mom_21"] = np.log(p["Oil_Close"] / p["Oil_Close"].shift(21))
        ratio_og = c / p["Oil_Close"]
        r_og = ratio_og.rolling(config["ratio_z_window"])
        p["Gold_Oil_Ratio_Z"] = (ratio_og - r_og.mean()) / (r_og.std() + 1e-10)

    # --- EUR/USD (dollar weakness from the cross-rate angle) ---
    if "EURUSD_Close" in p:
        p["EURUSD_Mom_21"] = np.log(p["EURUSD_Close"] / p["EURUSD_Close"].shift(21))
        eurusd_lr = np.log(p["EURUSD_Close"] / p["EURUSD_Close"].shift(1))
        eurusd_roll = eurusd_lr.rolling(config["vix_z_window"])
        p["EURUSD_Z"] = (eurusd_lr - eurusd_roll.mean()) / (eurusd_roll.std() + 1e-10)

    return p

In [ ]:
# 8 candidates — 4 original technicals + 4 macro factors with documented
# gold relationships. Section 4b narrows this to the top-7 by |corr| with
# direction (train-only) → 127 non-empty exhaustive subsets searched in §6b.
# GDX is available from 2006, so events start from ~2008 after the warmup window.
FEATURE_POOL = [
    # --- technicals (retained from original top performers) ---
    "Gold_Vol",              # realized 21-day volatility
    "Vol_Ratio",             # short/long vol ratio (regime)
    "VIX_Z",                 # VIX z-score (fear gauge)
    "Gold_SPX_Mom_Spread",   # gold vs S&P 500 21-day momentum spread
    # --- macro factors (new) ---
    "TIP_Z",                 # TIPS ETF return z-score (real yield proxy)
    "GDX_Gold_Ratio_Z",      # miner/gold ratio z-score (lead indicator)
    "EURUSD_Mom_21",         # EUR/USD 21-day momentum (dollar weakness)
    "Oil_Mom_21",            # crude oil 21-day momentum (commodity complex)
]
CANDIDATE_FEATURES = list(FEATURE_POOL)  # narrowed to top-7 in section 4b
FEATURES = list(CANDIDATE_FEATURES)      # replaced by the search winner in 6b

In [ ]:
featured = engineer_features(raw_data)
print(f"Engineered features; {len(FEATURE_POOL)}-feature pool (top-7 by |corr| selected in §4b).")

## 3. Three-Class Triple-Barrier Labeling

Every bar is an event (no directional pre-filter). For each entry we set two
volatility-scaled horizontal barriers and one vertical/time barrier, and label by
**whichever is touched first**:

- **+1 (up)** — the upper barrier `entry·(1 + pt_mult·σ)` is hit first.
- **−1 (down)** — the lower barrier `entry·(1 − sl_mult·σ)` is hit first.
- **0 (flat)** — neither is hit within `max_holding_days` (time barrier).

The model predicts this direction, then trades **long on +1, short on −1, flat on
0**. We store the **raw long return to the exit** (`ret`); a short's P&L is simply
`−ret`. Overlapping events are down-weighted by **average uniqueness**.

In [ ]:
def get_daily_vol(close, span):
    """EWMA standard deviation of daily log returns (fractional)."""
    return np.log(close / close.shift(1)).ewm(span=span).std()

In [ ]:
def apply_triple_barrier(close, events_idx, vol, config=CONFIG):
    """First-touched barrier per event -> 3-class direction label.

    Two-sided, volatility-scaled barriers (+pt_mult*sigma / -sl_mult*sigma) with a
    vertical barrier at max_holding_days. Label: +1 if the upper barrier is hit
    first, -1 if the lower, 0 if the time barrier wins. `ret` is the raw long
    return to the realized exit (a short earns -ret). Returns dir/ret/t1/holding.
    """
    vals = close.values
    idx = close.index
    pt, sl, max_h = config["pt_mult"], config["sl_mult"], config["max_holding_days"]
    rows = []
    for t0 in events_idx:
        i0 = idx.get_loc(t0)
        v = vol.loc[t0]
        if np.isnan(v) or v == 0:
            continue
        entry = vals[i0]
        up, dn = entry * (1 + pt * v), entry * (1 - sl * v)
        i_end = min(i0 + max_h, len(vals) - 1)
        exit_i, direction = i_end, 0          # default: time barrier -> flat
        for j in range(i0 + 1, i_end + 1):
            if vals[j] >= up:
                exit_i, direction = j, 1
                break
            if vals[j] <= dn:
                exit_i, direction = j, -1
                break
        ret = vals[exit_i] / entry - 1.0
        rows.append((t0, idx[exit_i], ret, exit_i - i0, direction))
    out = pd.DataFrame(rows, columns=["t0", "t1", "ret", "holding", "dir"]).set_index("t0")
    return out

In [ ]:
def average_uniqueness_weights(events, bar_index):
    """Average-uniqueness sample weights (de Prado): down-weight overlap."""
    i0 = bar_index.get_indexer(events.index)
    i1 = bar_index.get_indexer(events["t1"].values)
    # get_indexer returns -1 for dates not in bar_index; -1 silently creates
    # empty slices → NaN/inf weights. Fail loudly instead.
    assert (i0 >= 0).all(), f"{(i0 < 0).sum()} event entry dates missing from bar_index"
    assert (i1 >= 0).all(), f"{(i1 < 0).sum()} event exit (t1) dates missing from bar_index"
    count = np.zeros(len(bar_index))
    for a, b in zip(i0, i1):
        count[a:b + 1] += 1.0
    w = np.array([np.mean(1.0 / count[a:b + 1]) for a, b in zip(i0, i1)])
    return w / w.mean()                            # normalize to mean 1

In [ ]:
# Every bar with valid features + nonzero vol is an event (no directional filter)
close = featured["Gold_Close"]
daily_vol = get_daily_vol(close, CONFIG["vol_window"])

valid = featured[FEATURE_POOL].notna().all(axis=1) & daily_vol.notna()
events_idx = featured.index[valid]

events = apply_triple_barrier(close, events_idx, daily_vol, CONFIG)

# map direction {-1,0,+1} -> class {0,1,2} for the multiclass model
DIR_TO_CLASS = {-1: 0, 0: 1, 1: 2}
CLASS_TO_DIR = {0: -1, 1: 0, 2: 1}
events["y"] = events["dir"].map(DIR_TO_CLASS)
dist = events["dir"].value_counts(normalize=True).sort_index()
print(f"Generated {len(events)} labeled events.")
print(f"Direction mix  down: {dist.get(-1,0):.3f}  flat: {dist.get(0,0):.3f}  up: {dist.get(1,0):.3f}")

## 4. Assemble Dataset & Time-Ordered Holdout

Align features to event entry times, then carve off the most recent
`holdout_frac` of events as a final out-of-sample set the search never sees.

In [ ]:
# all candidate features available; section 4b narrows to top-7 by |corr|
X_all = featured.loc[events.index, FEATURE_POOL].copy()
y_all = events["y"].copy()                  # 3-class target {0,1,2}
ret_all = events["ret"].copy()              # raw long return to exit
dir_all = events["dir"].copy()              # realized first-touch direction

# label-end position within the EVENTS array (for purging in CV)
event_times = X_all.index.values
t1_pos_all = np.searchsorted(event_times, events["t1"].values)
assert not X_all.isna().any().any(), "NaNs leaked into the feature matrix"

In [ ]:
# time-ordered split into train / holdout
n = len(X_all)
split = int(n * (1 - CONFIG["holdout_frac"]))
X_tr, X_ho = X_all.iloc[:split], X_all.iloc[split:]
y_tr, y_ho = y_all.iloc[:split], y_all.iloc[split:]
ret_tr, ret_ho = ret_all.iloc[:split], ret_all.iloc[split:]
dir_tr, dir_ho = dir_all.iloc[:split], dir_all.iloc[split:]
t1pos_tr = t1_pos_all[:split]

# weights on training events only — featured.index as bar_index ensures all t1
# exit dates resolve (they come from close.index = featured.index), while passing
# only events.iloc[:split] means no holdout event window can inflate the
# concurrent-event count for any training bar.
w_tr = average_uniqueness_weights(events.iloc[:split], featured.index)

print(f"Train events: {len(X_tr)}  |  Holdout events: {len(X_ho)}")
print("Train dir mix: ", dir_tr.value_counts(normalize=True).round(3).to_dict())
print("Holdout dir mix:", dir_ho.value_counts(normalize=True).round(3).to_dict())

### 4b. Correlation Analysis

Computed on the **training split only** to avoid peeking at the holdout. We look
at (1) the feature-feature correlation matrix, (2) each feature's correlation
with the **signed direction** (−1/0/+1), and (3) any strongly correlated feature
pairs that signal redundancy.

In [ ]:
# feature-feature correlation heatmap
corr = X_tr.corr()
plt.figure(figsize=(14, 12))
sns.heatmap(corr, cmap="coolwarm", center=0, square=True,
            linewidths=0.5, cbar_kws={"shrink": 0.8})
plt.title("Feature Correlation Matrix (train)", fontweight="bold")
plt.tight_layout(); plt.show()

In [ ]:
# correlation of each feature with the signed direction target (-1/0/+1)
target_corr = (X_tr.corrwith(dir_tr.astype(float))
               .sort_values(key=lambda s: s.abs(), ascending=False))
plt.figure(figsize=(10, 8))
sns.barplot(x=target_corr.values, y=target_corr.index,
            palette=["green" if v > 0 else "red" for v in target_corr.values])
plt.axvline(0, color="black", lw=0.8)
plt.title("Feature Correlation with Direction (train)", fontweight="bold")
plt.xlabel("Pearson correlation with signed direction")
plt.tight_layout(); plt.show()
display(target_corr.to_frame("corr_with_direction").round(4))

In [ ]:
# Dynamically select the top-7 features by |corr| with direction (train only).
# This drops the weakest predictor from the 8-feature pool, reducing the
# exhaustive search from 255 to 127 subsets while keeping the best candidates.
CANDIDATE_FEATURES = list(target_corr.abs().nlargest(7).index)
X_tr = X_tr[CANDIDATE_FEATURES]
X_ho = X_ho[CANDIDATE_FEATURES]
print(f"Top-7 CANDIDATE_FEATURES (by |corr| with direction, train-only):")
print(CANDIDATE_FEATURES)
dropped = [f for f in FEATURE_POOL if f not in CANDIDATE_FEATURES]
print(f"Dropped: {dropped}")

In [ ]:
# strongly correlated feature pairs (potential redundancy / multicollinearity)
cc = corr.abs()
pairs = (cc.where(np.triu(np.ones(cc.shape), k=1).astype(bool))
         .stack().sort_values(ascending=False))
high = pairs[pairs > 0.8]
print("Feature pairs with |corr| > 0.8:")
display(high.to_frame("abs_corr").round(3) if len(high) else "(none)")

## 5. Purged Walk-Forward Cross-Validation

Expanding walk-forward splits where, for each test fold, training samples whose
label window overlaps the test period (plus an `embargo`) are **purged** —
eliminating look-ahead from overlapping triple-barrier labels.

In [ ]:
class PurgedWalkForwardCV:
    """Expanding walk-forward splits with purging + embargo (positional)."""
    def __init__(self, n_splits, t1_pos, embargo=0):
        self.n_splits = n_splits
        self.t1_pos = np.asarray(t1_pos)
        self.embargo = embargo

    def split(self, X):
        n = len(X)
        indices = np.arange(n)
        # test folds = blocks 1..n_splits (block 0 is the initial training seed)
        test_folds = np.array_split(indices, self.n_splits + 1)[1:]
        for test_idx in test_folds:
            test_start = test_idx[0]
            train_idx = indices[:test_start]
            # purge train samples whose label extends to/after (test_start-embargo)
            keep = self.t1_pos[train_idx] < (test_start - self.embargo)
            train_idx = train_idx[keep]
            if len(train_idx) == 0:
                continue
            yield train_idx, test_idx

    def get_n_splits(self, X=None, y=None, groups=None):
        return self.n_splits

In [ ]:
cv = PurgedWalkForwardCV(CONFIG["n_splits"], t1pos_tr, CONFIG["embargo"])

# sanity check: no train label-end bleeds past the embargo into any test fold
for tr, te in cv.split(X_tr):
    assert t1pos_tr[tr].max() < te[0] - CONFIG["embargo"]
print(f"{cv.get_n_splits()} purged walk-forward folds constructed.")

## 6. Multiclass Model + Profit Engine

A multiclass (`down/flat/up`) XGBoost. Everything downstream is scored by a single
**$-backtest engine** (`simulate_from_edges`): walk events in time order, and when
flat take a **non-overlapping** position whose direction is the sign of the
**edge** `= P(up) − P(down)` (if `|edge| > tau`), sized **confidence-proportionally**
`frac = min(1, size_scale·|edge|)` of *current* equity, compounding P&L net of
round-trip cost. The objective everywhere is **final account value** (profit).

In [ ]:
import optuna
from xgboost import XGBClassifier
optuna.logging.set_verbosity(optuna.logging.WARNING)

def make_xgb(**params):
    """Multiclass (3-class) XGBoost with consistent fixed settings.
    nthread=1 forces single-threaded execution for full reproducibility across
    runs (parallel float-point ordering is otherwise nondeterministic)."""
    return XGBClassifier(**params, objective="multi:softprob", num_class=3,
                         eval_metric="mlogloss", tree_method="hist",
                         random_state=CONFIG["random_state"], nthread=1)

def fit_with_es(params, X, y, w, tr):
    """Fit on fold-train indices `tr`, using a time-tail slice for early stopping
    (the validation tail precedes the test fold in time, so no look-ahead)."""
    # validation tail = max(50, 15%) of the fold: keeps the usual 15% on large
    # folds but guarantees >=50 samples for reliable early stopping on small
    # early folds (min picks the smaller k -> the larger tr[k:] tail).
    k = min(int(len(tr) * 0.85), len(tr) - 50)
    fit_idx, val_idx = tr[:k], tr[k:]
    if k <= 0 or len(val_idx) == 0:
        m = make_xgb(**params)
        m.fit(X.iloc[tr], y.iloc[tr], sample_weight=w[tr])
        return m
    m = make_xgb(**params, early_stopping_rounds=50)
    m.fit(X.iloc[fit_idx], y.iloc[fit_idx], sample_weight=w[fit_idx],
          eval_set=[(X.iloc[val_idx], y.iloc[val_idx])], verbose=False)
    return m

def directional_edge(proba):
    """P(up) - P(down) from multiclass proba columns [down, flat, up]."""
    return proba[:, 2] - proba[:, 0]

In [ ]:
def simulate_from_edges(entry_times, edge, tau, scale, config=CONFIG,
                        return_trades=False):
    """Compound a $-account over time-ordered events (one position at a time).

    direction = sign(edge) when |edge| > tau (long on +, short on -); size =
    min(max_position_frac, scale*|edge|) of CURRENT equity (no leverage). A short
    earns -ret. Round-trip cost charged on traded notional. Returns final equity
    (and the trade ledger if requested).

    Reads the module-level `events` DataFrame for t1/ret lookup; entry_times must
    be a subset of events.index (guaranteed for X_tr.index and X_ho.index).
    """
    entry_times = pd.DatetimeIndex(entry_times)
    sub = events.reindex(entry_times)
    if sub["t1"].isna().any():
        raise ValueError(
            f"{sub['t1'].isna().sum()} entry_times not found in events.index — "
            "this would silently disable the non-overlap guard via NaT comparison."
        )
    t1s, rets = sub["t1"], sub["ret"].values
    cap = config["start_capital"]
    last_exit = None
    maxfrac, cost = config["max_position_frac"], config["cost_per_trade"]
    trades = []
    for k in range(len(entry_times)):
        t0 = entry_times[k]
        if last_exit is not None and t0 <= last_exit:   # still holding a position
            continue
        e = edge[k]
        direction = 1 if e > tau else (-1 if e < -tau else 0)
        if direction == 0:
            continue
        frac = min(maxfrac, scale * abs(e))
        signed = direction * rets[k]
        cap = cap * (1 + frac * signed) - cap * frac * cost * 2.0
        cap = max(cap, 0.0)          # bankruptcy floor (no negative equity)
        last_exit = t1s.iloc[k]
        if return_trades:
            trades.append((t0, t1s.iloc[k], direction, frac, signed, rets[k], cap))
    if return_trades:
        cols = ["t0", "t1", "dir", "frac", "signed_ret", "ret", "equity"]
        return cap, pd.DataFrame(trades, columns=cols).set_index("t0")
    return cap

In [ ]:
def oof_edges(params, Xsub):
    """Out-of-fold directional edges on the training set (leak-free)."""
    oof = np.full(len(y_tr), np.nan)
    for tr, te in cv.split(Xsub):
        m = fit_with_es(params, Xsub, y_tr, w_tr, tr)
        oof[te] = directional_edge(m.predict_proba(Xsub.iloc[te]))
    return oof

def trades_to_sharpe(trades, config=CONFIG):
    """Annualized Sharpe from a trades ledger via a daily equity curve.

    Spreads each trade's compounded multiplier uniformly across its holding days
    (using featured.index as the business-day calendar) then computes annualized
    Sharpe. Returns 0.0 for empty or single-trade ledgers.
    """
    if len(trades) < 2:
        return 0.0
    t_start, t_end = trades.index.min(), trades["t1"].max()
    all_dates = featured.index[(featured.index >= t_start) & (featured.index <= t_end)]
    if len(all_dates) < 2:
        return 0.0
    factor = pd.Series(1.0, index=all_dates)
    cap_prev = config["start_capital"]
    for t0, row in trades.iterrows():
        M = row["equity"] / cap_prev
        cap_prev = row["equity"]
        span = featured.index[(featured.index >= t0) & (featured.index <= row["t1"])]
        span = span[span.isin(all_dates)]
        if len(span) == 0:
            continue
        factor.loc[span] *= M ** (1.0 / len(span))
    daily = config["start_capital"] * factor.cumprod()
    ret = daily.pct_change().fillna(0)
    return ret.mean() / (ret.std() + 1e-12) * np.sqrt(config["trading_days"])

def simulate_composite(entry_times, edge, tau, scale, config=CONFIG):
    """Combined profit × Sharpe objective: profit_ratio * annualized_Sharpe.

    Both must be positive to score well. A high-profit but volatile path and a
    low-drawdown but unprofitable path both score below a balanced strategy.
    """
    final_eq, trades = simulate_from_edges(entry_times, edge, tau, scale, config,
                                           return_trades=True)
    sharpe = trades_to_sharpe(trades, config)
    profit_ratio = final_eq / config["start_capital"]
    # floor Sharpe at 0.1 so folds with very few trades (Sharpe ≈ noise) don't
    # score 0 and bias Optuna toward low-tau / high-trade-count settings.
    return profit_ratio * max(sharpe, 0.1)

def cv_objective(params, Xsub, tau=None, scale=None):
    """OOF profit × Sharpe composite (the search objective for all tuning stages)."""
    tau = CONFIG["edge_threshold"] if tau is None else tau
    scale = CONFIG["size_scale"] if scale is None else scale
    oof = oof_edges(params, Xsub)
    mask = ~np.isnan(oof)
    return simulate_composite(Xsub.index[mask], oof[mask], tau, scale)

## 6b. Feature-Combo Search (maximize profit × Sharpe)

Search subsets of the 7 candidates for the combination with the highest
out-of-fold **profit × Sharpe composite** (fixed quick hyperparameters). We run
an **exhaustive** sweep of all 127 non-empty subsets — the gold standard — and a
**greedy forward** search for comparison. Greedy judges each feature by its
*marginal* gain, so it can
miss pairs that only help together; we therefore lock `FEATURES` to the exhaustive
winner and report whether greedy agreed.

In [ ]:
import itertools

QUICK_PARAMS = dict(n_estimators=200, max_depth=3, learning_rate=0.05,
                    subsample=0.8, colsample_bytree=0.8, min_child_weight=5,
                    reg_alpha=0.01, reg_lambda=0.1)
# Snapshot reference: the X_tr[FEATURES] rebinding in the lock cell below will
# NOT affect _X_search, so partial notebook reruns still search the full 7-column
# matrix rather than crashing with a KeyError on features outside the winner.
_X_search = X_tr

def subset_profit(subset):
    """OOF profit for a feature subset (quick fixed params).

    Pure profit is used here (not the composite) because QUICK_PARAMS produce
    a weak model whose OOF Sharpe estimate has high variance — adding Sharpe to
    the feature-search objective would amplify noise rather than improve ranking.
    The composite (profit × Sharpe) is applied in Optuna and tau/scale tuning
    where the model is properly fitted.
    """
    oof = oof_edges(QUICK_PARAMS, _X_search[list(subset)])
    mask = ~np.isnan(oof)
    return simulate_from_edges(_X_search.index[mask], oof[mask],
                               CONFIG["edge_threshold"], CONFIG["size_scale"])

In [ ]:
# exhaustive: every non-empty subset of the 7 candidates
exhaustive = [(combo, subset_profit(combo))
              for r in range(1, len(CANDIDATE_FEATURES) + 1)
              for combo in itertools.combinations(CANDIDATE_FEATURES, r)]
exh_best, exh_val = max(exhaustive, key=lambda kv: kv[1])
print(f"Exhaustive: tested {len(exhaustive)} subsets.")
print(f"  winner (OOF equity=${exh_val:,.0f}): {list(exh_best)}")
print("  top 3:")
for combo, v in sorted(exhaustive, key=lambda kv: -kv[1])[:3]:
    print(f"    ${v:,.0f}  {list(combo)}")

In [ ]:
# greedy forward selection by profit (add the best-improving feature until none helps)
remaining, selected, greedy_val = list(CANDIDATE_FEATURES), [], -np.inf
while remaining:
    # evaluate each candidate once; capture both winner and its score together
    # (avoids a redundant second call that would re-run the full OOF CV)
    cand, val = max(
        ((f, subset_profit(selected + [f])) for f in remaining), key=lambda x: x[1]
    )
    if val <= greedy_val:
        break
    selected.append(cand); remaining.remove(cand); greedy_val = val
print(f"Greedy winner (OOF equity=${greedy_val:,.0f}): {selected}")
print(f"Greedy == exhaustive subset? {set(selected) == set(exh_best)}"
      f"  (OOF equity gap ${exh_val - greedy_val:,.0f})")

In [ ]:
# lock the feature set to the exhaustive winner and subset the design matrices
FEATURES = list(exh_best)
X_tr, X_ho = X_tr[FEATURES], X_ho[FEATURES]
print(f"FEATURES = {FEATURES}")

## 6c. Hyperparameter Search (Optuna, profit × Sharpe objective)

Optuna maximizes the **out-of-fold profit × Sharpe composite** — the product of
the profit ratio (final equity ÷ start capital) and the annualized Sharpe on the
OOF equity curve. Both profit and risk-adjusted quality must be high to score
well. Sample weights are passed to every fit.

In [ ]:
def objective(trial):
    params = {
        "n_estimators": trial.suggest_int("n_estimators", 200, 600, step=100),
        "max_depth": trial.suggest_int("max_depth", 2, 5),
        "learning_rate": trial.suggest_float("learning_rate", 0.005, 0.1, log=True),
        "subsample": trial.suggest_float("subsample", 0.6, 1.0),
        "colsample_bytree": trial.suggest_float("colsample_bytree", 0.6, 1.0),
        "min_child_weight": trial.suggest_int("min_child_weight", 1, 10),
        "reg_alpha": trial.suggest_float("reg_alpha", 1e-3, 5.0, log=True),
        "reg_lambda": trial.suggest_float("reg_lambda", 1e-3, 5.0, log=True),
    }
    return cv_objective(params, X_tr)

In [ ]:
study = optuna.create_study(
    direction="maximize",
    sampler=optuna.samplers.TPESampler(seed=CONFIG["random_state"]))
study.optimize(objective, n_trials=CONFIG["n_trials"], show_progress_bar=False)

best_params = study.best_params
print(f"Best OOF profit×Sharpe: {study.best_value:.4f}")
print("Best params:", best_params)

## 7. Threshold & Sizing Tuning

Grid-search the **edge threshold** `tau` and **size scale** on the leak-free OOF
edges (best params) to maximize the **profit × Sharpe composite**, then fit the
final multiclass model on all training data. (No probability calibration: it is
binary-oriented and adds nothing to this objective.)

In [ ]:
# OOF edges with the tuned hyperparameters
oof = oof_edges(best_params, X_tr)
oof_mask = ~np.isnan(oof)
oof_times = X_tr.index[oof_mask]

In [ ]:
# grid-search (tau, size_scale) for maximum OOF profit×Sharpe composite
# tau: 0.08..0.50 in steps of 0.02 (22 points); scale: 9 values -> 198 combos
best_tau, best_scale, best_val = CONFIG["edge_threshold"], CONFIG["size_scale"], -np.inf
for tau in np.arange(0.08, 0.52, 0.02):
    for scale in (0.5, 1.0, 2.0, 3.0, 5.0, 8.0, 12.0, 20.0, 30.0):
        v = simulate_composite(oof_times, oof[oof_mask], tau, scale)
        if v > best_val:
            best_val, best_tau, best_scale = v, tau, scale
print(f"Tuned tau={best_tau:.3f}, size_scale={best_scale:.1f}  ->  OOF composite={best_val:.4f}")

In [ ]:
# final multiclass model — reuse fit_with_es so early stopping mirrors the
# CV folds that produced best_params (avoids training more trees than CV used)
xgb_best = fit_with_es(best_params, X_tr, y_tr, w_tr, np.arange(len(X_tr)))

## 8. Out-of-Sample Evaluation

Three-class metrics on the untouched holdout. Accuracy is shown against the
majority-class baseline (the right reference for an imbalanced 3-class target).

In [ ]:
from sklearn.metrics import (accuracy_score, roc_auc_score, classification_report,
                             confusion_matrix)

proba_ho = xgb_best.predict_proba(X_ho)                # columns [down, flat, up]
pred_ho = proba_ho.argmax(axis=1)

majority_acc = y_ho.value_counts(normalize=True).max()
print(f"Accuracy:            {accuracy_score(y_ho, pred_ho):.4f}")
print(f"Majority-class acc:  {majority_acc:.4f}")
print(f"Macro ROC-AUC (ovr): {roc_auc_score(y_ho, proba_ho, multi_class='ovr', average='macro'):.4f}\n")
print(classification_report(y_ho, pred_ho, target_names=["down", "flat", "up"]))

In [ ]:
cm = confusion_matrix(y_ho, pred_ho, labels=[0, 1, 2])
plt.figure(figsize=(6, 5))
sns.heatmap(cm, annot=True, fmt="d", cmap="Blues", cbar=False,
            xticklabels=["down", "flat", "up"], yticklabels=["down", "flat", "up"])
plt.title("Confusion Matrix (holdout)", fontweight="bold")
plt.xlabel("Predicted direction"); plt.ylabel("Actual direction")
plt.tight_layout(); plt.show()

## 9. Model Comparison (by profit) + SHAP

Benchmark the tuned XGBoost against multiclass LightGBM, a scaled multinomial
logistic baseline, and an **LSTM sequence model** — each scored by the **$ profit**
its directional edges generate on the holdout (same `tau`/`size_scale`), plus
macro AUC and accuracy. Then explain the XGBoost model with SHAP (up-class).

In [ ]:
from lightgbm import LGBMClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
import shap

def evaluate(name, est, fit_kwargs=None, prefit=False):
    """Score a multiclass model on the holdout by the profit of its edges."""
    fit_kwargs = fit_kwargs or {}
    if not prefit:                                   # xgb_best already fit in section 7
        est.fit(X_tr, y_tr, **fit_kwargs)
    proba = est.predict_proba(X_ho)                  # [down, flat, up]
    edge = directional_edge(proba)
    final_eq, ledger = simulate_from_edges(X_ho.index, edge, best_tau, best_scale,
                                           return_trades=True)
    return {"Model": name,
            "AUC": roc_auc_score(y_ho, proba, multi_class="ovr", average="macro"),
            "Accuracy": accuracy_score(y_ho, proba.argmax(axis=1)),
            "Final $": round(final_eq),
            "Profit $": round(final_eq - CONFIG["start_capital"]),
            "Trades": len(ledger)}

In [ ]:
# tabular models
rows = [
    evaluate("XGBoost (tuned)", xgb_best, prefit=True),   # fitted in section 7
    evaluate("LightGBM", LGBMClassifier(
        objective="multiclass", num_class=3,
        n_estimators=400, max_depth=4, learning_rate=0.02,
        subsample=0.8, colsample_bytree=0.8,
        random_state=CONFIG["random_state"], n_jobs=-1, verbose=-1),
        {"sample_weight": w_tr}),
    evaluate("Logistic (scaled)", make_pipeline(
        StandardScaler(),
        LogisticRegression(max_iter=1000, C=0.5,
                           random_state=CONFIG["random_state"])),
        {"logisticregression__sample_weight": w_tr}),
]

In [ ]:
# LSTM sequence-model baseline (skipped gracefully if TensorFlow unavailable)
try:
    import tensorflow as tf
    from tensorflow.keras import layers, models
    tf.random.set_seed(CONFIG["random_state"])

    seq_len = CONFIG["lstm_seq_len"]
    feat_all = featured[FEATURES].astype(float)
    sc = StandardScaler().fit(feat_all.loc[:X_tr.index[-1]].values)   # train-only scaling
    scaled = pd.DataFrame(sc.transform(feat_all.values),
                          index=feat_all.index, columns=FEATURES).fillna(0.0)
    arr = scaled.values
    pos = {t: i for i, t in enumerate(scaled.index)}

    def make_seqs(event_index):
        seqs, kept = [], []
        for t in event_index:
            i = pos[t]
            if i - seq_len + 1 >= 0:
                seqs.append(arr[i - seq_len + 1:i + 1])
                kept.append(t)
        return np.asarray(seqs), pd.Index(kept)

    Xs_tr, idx_tr = make_seqs(X_tr.index)
    Xs_ho, idx_ho = make_seqs(X_ho.index)
    y_seq = y_tr.loc[idx_tr].values
    w_seq = pd.Series(w_tr, index=X_tr.index).loc[idx_tr].values

    lstm = models.Sequential([
        layers.Input((seq_len, len(FEATURES))),
        layers.LSTM(32),
        layers.Dropout(0.3),
        layers.Dense(16, activation="relu"),
        layers.Dense(3, activation="softmax"),
    ])
    lstm.compile(optimizer="adam", loss="sparse_categorical_crossentropy")
    lstm.fit(Xs_tr, y_seq, sample_weight=w_seq,
             epochs=CONFIG["lstm_epochs"], batch_size=64, verbose=0)

    proba = lstm.predict(Xs_ho, verbose=0)
    edge = directional_edge(proba)
    final_eq, ledger = simulate_from_edges(idx_ho, edge, best_tau, best_scale,
                                           return_trades=True)
    rows.append({"Model": "LSTM (sequence)",
                 "AUC": roc_auc_score(y_ho.loc[idx_ho], proba, multi_class="ovr", average="macro"),
                 "Accuracy": accuracy_score(y_ho.loc[idx_ho], proba.argmax(axis=1)),
                 "Final $": round(final_eq),
                 "Profit $": round(final_eq - CONFIG["start_capital"]),
                 "Trades": len(ledger)})
except Exception as e:
    print("LSTM baseline skipped:", repr(e))

In [ ]:
comparison = pd.DataFrame(rows).set_index("Model").round(4)
print("Majority-class accuracy:", round(majority_acc, 4))
display(comparison)

In [ ]:
# SHAP explainability for the tuned XGBoost (up-class contributions)
explainer = shap.TreeExplainer(xgb_best)
sv = explainer.shap_values(X_ho)
sv_up = sv[2] if isinstance(sv, list) else sv[..., 2]
shap.summary_plot(sv_up, X_ho, show=True)

## 10. $100,000 Profit Backtest

Run the profit engine on the holdout with the tuned `tau`/`size_scale`: the model
picks direction and size for each **non-overlapping** trade, compounding a
**$100,000** account with **no leverage**. We report the **final account value**
and total profit first, then risk stats (Sharpe / Sortino / max-DD / Calmar) for
context, trade stats, the equity curve, and a per-regime breakdown.

In [ ]:
# directional edges on the holdout, then run the compounding $-engine
edge_ho = directional_edge(proba_ho)
final_equity, trades = simulate_from_edges(X_ho.index, edge_ho, best_tau, best_scale,
                                           return_trades=True)
start = CONFIG["start_capital"]
print(f"Start:  ${start:,.0f}")
print(f"Final:  ${final_equity:,.0f}")
print(f"Profit: ${final_equity - start:,.0f}  ({final_equity/start - 1:.1%})")
print(f"Trades: {len(trades)}  (long {int((trades['dir']==1).sum())}, "
      f"short {int((trades['dir']==-1).sum())})")

In [ ]:
# daily equity curve: distribute each trade's exact multiplier across its holding
# days so the compounded curve reproduces the engine's final value precisely.
ho_dates = featured.loc[X_ho.index[0]:X_ho.index[-1]].index
factor = pd.Series(1.0, index=ho_dates)
cap_prev = start
for t0, row in trades.iterrows():
    M = row["equity"] / cap_prev          # net multiplier for this trade (incl. costs)
    cap_prev = row["equity"]
    span = featured.loc[t0:row["t1"]].index
    span = span[span.isin(ho_dates)]
    if len(span) == 0:
        continue
    factor.loc[span] *= M ** (1.0 / len(span))
strat_eq = start * factor.cumprod()
# Note: M**(1/n) spreading assumes uniform daily growth within each trade.
# Intra-trade price paths are unobserved at daily bar resolution, so Sharpe
# and max-drawdown below reflect an approximation — actual intra-trade
# drawdowns may be deeper than reported.
bench_eq = start * (1 + close.loc[ho_dates].pct_change().fillna(0)).cumprod()

In [ ]:
def perf_stats(eq):
    daily = eq.pct_change().fillna(0)
    td = CONFIG["trading_days"]
    total = eq.iloc[-1] / eq.iloc[0] - 1
    ann = (1 + total) ** (td / len(daily)) - 1
    sharpe = daily.mean() / (daily.std() + 1e-12) * np.sqrt(td)
    downside = daily[daily < 0].std()
    sortino = daily.mean() / (downside + 1e-12) * np.sqrt(td)
    dd = (eq / eq.cummax() - 1).min()
    calmar = ann / abs(dd) if dd != 0 else np.nan
    return {"Final $": round(eq.iloc[-1]), "Total Return": total, "Annualized": ann,
            "Sharpe": sharpe, "Sortino": sortino, "Max Drawdown": dd, "Calmar": calmar}

perf = pd.DataFrame({"Strategy": perf_stats(strat_eq),
                     "Buy & Hold": perf_stats(bench_eq)})

In [ ]:
# equity & drawdown plots
fig, ax = plt.subplots(2, 1, figsize=(14, 9), sharex=True,
                       gridspec_kw={"height_ratios": [3, 1]})
ax[0].plot(bench_eq.index, bench_eq, color="gray", alpha=0.8, label="Buy & Hold Gold")
ax[0].plot(strat_eq.index, strat_eq, color="green", lw=2, label="Model (long/short, sized)")
ax[0].axhline(start, color="black", lw=0.7, ls=":")
ax[0].set_title("Out-of-Sample Account Value (net of costs)", fontweight="bold")
ax[0].set_ylabel("Account value ($)"); ax[0].legend(loc="upper left")
ax[1].fill_between(strat_eq.index, strat_eq / strat_eq.cummax() - 1, color="red", alpha=0.4)
ax[1].set_ylabel("Drawdown"); ax[1].set_xlabel("Date")
plt.tight_layout(); plt.show()

In [ ]:
display(perf.round(4))

In [ ]:
# trade-level stats (P&L in the direction actually taken)
if trades.empty:
    print("No trades taken on holdout — trade stats unavailable.")
else:
    pnl = trades["dir"] * trades["ret"]
    wins, losses = pnl[pnl > 0], pnl[pnl <= 0]
    profit_factor = wins.sum() / abs(losses.sum()) if len(losses) and losses.sum() != 0 else np.inf
    holding_days = [(t1 - t0).days for t0, t1 in zip(trades.index, trades["t1"])]
    print(f"Hit rate:       {(pnl > 0).mean():.2%}")
    print(f"Avg win / loss: {wins.mean():.4f} / {losses.mean():.4f}")
    print(f"Profit factor:  {profit_factor:.2f}")
    print(f"Avg position:   {trades['frac'].mean():.1%} of equity")
    print(f"Avg holding:    {np.mean(holding_days):.1f} calendar days")

In [ ]:
# per-regime breakdown (regime measured at entry)
if trades.empty:
    print("No trades taken on holdout — regime breakdown unavailable.")
else:
    reg = featured.loc[trades.index]
    trades_reg = trades.assign(
        pnl=trades["dir"] * trades["ret"],
        Vol_Regime=np.where(reg["High_Vol_Regime"] == 1, "High Vol", "Low Vol"),
        Trend=np.where(reg["Trend_Sign"] == 1, "Uptrend", "Downtrend"))
    breakdown = (trades_reg.groupby(["Vol_Regime", "Trend"])
                 .agg(Trades=("pnl", "size"), Avg_PnL=("pnl", "mean"),
                      Hit_Rate=("pnl", lambda s: (s > 0).mean()))
                 .round(4))
    display(breakdown)